# LeetCode 2170 — Minimum Operations to Make the Array Alternating bằng Python

Link đề: https://leetcode.com/problems/minimum-operations-to-make-the-array-alternating/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Greedy, Counting

Skeleton LeetCode:

```python
class Solution:
    def minimumOperations(self, nums: List[int]) -> int:
        ...
```

Mảng "xen kẽ" = các vị trí chẵn cùng một giá trị `x`, các vị trí lẻ cùng một giá trị `y`, và `x ≠ y`. Số thao tác = số phần tử **không** khớp. Bài thu về: **chọn `x` cho nhóm chẵn, `y` cho nhóm lẻ, `x ≠ y`, sao cho số phần tử giữ lại nhiều nhất**. Notebook đi từ **thử mọi cặp `(x, y)`** → **đếm tần suất + thử mọi `x` với `y` tốt nhất khác `x`** → **chỉ cần top-2 mỗi nhóm** → **sắp xếp tần suất** → **một lượt đếm, top-2 không cần sắp**.

## 1. Mục tiêu học

1. Đưa điều kiện "xen kẽ" về dạng **hai giá trị** `x` (vị trí chẵn) và `y` (vị trí lẻ) với `x ≠ y`.
2. Chuyển "số thao tác nhỏ nhất" thành "số phần tử **giữ lại** lớn nhất" — `n − (cntEven[x] + cntOdd[y])`.
3. Chứng minh chỉ cần **hai giá trị phổ biến nhất** của mỗi nhóm.
4. Xử lý trường hợp **giá trị phổ biến nhất trùng nhau** ở hai nhóm, và các nhóm chỉ có một giá trị (top-2 có tần suất 0).
5. Kiểm thử đối chiếu với vét cạn trên mảng ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Cho mảng `nums` (đánh số từ 0) gồm `n` số nguyên dương. Mảng là **xen kẽ** nếu:
- `nums[i − 2] == nums[i]` với `2 ≤ i ≤ n − 1`;
- `nums[i − 1] != nums[i]` với `1 ≤ i ≤ n − 1`.

Một thao tác: chọn chỉ số `i` và đổi `nums[i]` thành **số nguyên dương bất kỳ**. Trả về số thao tác nhỏ nhất để mảng xen kẽ.

**Ví dụ 1:** `[3,1,3,2,4,3]` → `3` (thành `[3,1,3,1,3,1]`).

**Ví dụ 2:** `[1,2,2,2,2]` → `2` (thành `[1,2,1,2,1]`; **không** được thành `[2,2,2,2,2]` vì hai phần tử kề nhau bằng nhau).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ nums[i] ≤ 10⁵`.

In [1]:
import random
import time
from collections import Counter
from typing import List

examples = [([3, 1, 3, 2, 4, 3], 3), ([1, 2, 2, 2, 2], 2)]

for nums, exp in examples:
    print(f"nums={nums}: vị trí chẵn {nums[0::2]}, vị trí lẻ {nums[1::2]}, kỳ vọng {exp}")

nums=[3, 1, 3, 2, 4, 3]: vị trí chẵn [3, 3, 4], vị trí lẻ [1, 2, 3], kỳ vọng 3
nums=[1, 2, 2, 2, 2]: vị trí chẵn [1, 2, 2], vị trí lẻ [2, 2], kỳ vọng 2


## 3. Kiến thức nền

### 3.1 Cắt lát với bước nhảy

`nums[0::2]` lấy các vị trí chẵn, `nums[1::2]` lấy các vị trí lẻ.

### 3.2 `Counter.most_common(k)`

In [2]:
c = Counter([3, 3, 4, 1, 3, 4])
print(c.most_common(2), "| most_common trên Counter rỗng:", Counter().most_common(2))

[(3, 3), (4, 2)] | most_common trên Counter rỗng: []


### 3.3 Đệm top-2 bằng "giá trị giả" tần suất 0

Khi một nhóm chỉ có một giá trị (hoặc rỗng — `n = 1` thì nhóm lẻ rỗng), ta đệm thêm `(None, 0)` hoặc `(0, 0)` để luôn có đủ 2 phần tử. Giá trị `0` an toàn vì đề chỉ có số **dương**, và ta có thể đổi một phần tử thành số dương "mới" bất kỳ — "giữ 0 phần tử" luôn khả thi.

In [3]:
def top2(counter):
    best = counter.most_common(2)
    while len(best) < 2:
        best.append((0, 0))          # 0 không xuất hiện trong nums (số dương)
    return best

print(top2(Counter([5, 5, 5])), top2(Counter()))

[(5, 3), (0, 0)] [(0, 0), (0, 0)]


### 3.4 Vì sao luôn có giá trị "mới" để gán?

Đề cho đổi thành **bất kỳ** số dương; luôn tồn tại số dương không xuất hiện trong mảng (ví dụ `max(nums) + 1`). Nên nếu tối ưu là "không giữ phần tử nào" ở một nhóm, ta gán cả nhóm thành một số mới khác giá trị của nhóm kia.

## 4. Ý tưởng nền tảng / trực giác

Điều kiện 1 ⇔ mọi vị trí chẵn bằng nhau (gọi là `x`) và mọi vị trí lẻ bằng nhau (`y`). Điều kiện 2 ⇔ `x ≠ y` (khi `n ≥ 2`).

Với `(x, y)` cố định, số thao tác = (số vị trí chẵn ≠ x) + (số vị trí lẻ ≠ y) = `n − cntE[x] − cntO[y]`.

Tính tay ví dụ 1: chẵn `[3, 3, 4]` ⇒ `cntE = {3: 2, 4: 1}`; lẻ `[1, 2, 3]` ⇒ `cntO = {1: 1, 2: 1, 3: 1}`. Chọn `x = 3` (giữ 2). `y` phải `≠ 3` ⇒ `y = 1` hoặc `2` (giữ 1). Giữ `3` ⇒ thao tác `6 − 3 = 3` ✓.

Ví dụ 2: chẵn `[1, 2, 2]` ⇒ `{2: 2, 1: 1}`; lẻ `[2, 2]` ⇒ `{2: 2}`. Hai nhóm đều muốn `2` — **xung đột**. Lựa chọn: `x = 2, y = (mới)` giữ `2 + 0`; hoặc `x = 1, y = 2` giữ `1 + 2 = 3` ⇒ thao tác `5 − 3 = 2` ✓.

In [4]:
for nums, _ in examples:
    cE, cO = Counter(nums[0::2]), Counter(nums[1::2])
    print(f"{nums}: cntE={dict(cE)}, cntO={dict(cO)}")

[3, 1, 3, 2, 4, 3]: cntE={3: 2, 4: 1}, cntO={1: 1, 2: 1, 3: 1}
[1, 2, 2, 2, 2]: cntE={1: 1, 2: 2}, cntO={2: 2}


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp `(x, y)`

Ứng viên cho `x`, `y`: mọi giá trị xuất hiện trong mảng, cộng thêm **hai giá trị mới** (đại diện cho "gán thành số chưa có"; cần hai số mới khác nhau để `x ≠ y` khi cả hai đều mới). Với mỗi cặp `x ≠ y`, đếm số thao tác trực tiếp.

In [5]:
def min_ops_brute(nums):
    n = len(nums)
    if n == 1:
        return 0
    m = max(nums)
    cand = set(nums) | {m + 1, m + 2}
    best = n
    for x in cand:
        for y in cand:
            if x == y:
                continue
            ops = sum(1 for i in range(n) if nums[i] != (x if i % 2 == 0 else y))
            best = min(best, ops)
    return best

for nums_, exp in examples:
    print(min_ops_brute(nums_), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )


**Độ phức tạp:** `O(V² · n)` với `V` = số giá trị khác nhau — `10¹⁵` trong trường hợp xấu. **Nút thắt:** (1) đếm lại mỗi cặp — thay bằng `cntE[x] + cntO[y]`; (2) thử mọi cặp — chỉ cần top-2.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề top-2.** Gọi `(a₁, f₁), (a₂, f₂)` là hai giá trị phổ biến nhất của nhóm chẵn (`f₁ ≥ f₂`), `(b₁, g₁), (b₂, g₂)` của nhóm lẻ. Tối ưu "giữ lại" là:
- **nếu** `a₁ ≠ b₁` **thì** `f₁ + g₁` (cả hai nhóm lấy giá trị tốt nhất của mình);
- **ngược lại** `max(f₁ + g₂, f₂ + g₁)`.

**Vì sao đủ?** Với mọi cặp `x ≠ y`: `cntE[x] ≤ f₁`, `cntO[y] ≤ g₁`. Nếu `a₁ ≠ b₁` thì cận `f₁ + g₁` đạt được. Nếu `a₁ = b₁`: hoặc `x ≠ a₁` (khi đó `cntE[x] ≤ f₂`, tổng `≤ f₂ + g₁`), hoặc `x = a₁` thì `y ≠ a₁ = b₁` (khi đó `cntO[y] ≤ g₂`, tổng `≤ f₁ + g₂`). Cả hai cận đều đạt được.

Đáp án `= n − giữ lại`.

### 6.1 Vì sao "top-2" là đủ ở đây — và khi nào **không** đủ

Lập luận top-2 dựa vào việc mỗi nhóm chỉ có **một** nhóm "hàng xóm" phải khác giá trị: hàng xóm chỉ "chặn" được **một** giá trị, nên giá trị tốt nhất hoặc nhì luôn còn dùng được. Nếu một nhóm có **hai** hàng xóm (ví dụ chuỗi 3 lớp `i mod 3` với lớp giữa kề cả hai lớp kia), hai hàng xóm có thể chặn cả top-1 lẫn top-2 ⇒ cần **top-3**. Cell dưới dựng phản ví dụ: lớp giữa có tần suất `A:3, B:3, C:2`, hai lớp ngoài "muốn" `A` và `B` rất mạnh.

In [6]:
def chain_cost(groups, choice):
    return sum(sum(1 for v in g if v != c) for g, c in zip(groups, choice))

def chain_best(groups, topk):
    NEW = [-1, -2]
    prev = None
    for g in groups:
        c = Counter(g); cand = [v for v, _ in c.most_common(topk)] + NEW
        cur = {}
        for v in cand:
            cost = len(g) - c.get(v, 0)
            cur[v] = cost if prev is None else cost + min(pc for pv, pc in prev.items() if pv != v)
        prev = cur
    return min(prev.values())

groups = [["A"] * 5, ["A"] * 3 + ["B"] * 3 + ["C"] * 2, ["B"] * 5]
print("top-2 mỗi lớp:", chain_best(groups, 2), "| top-3 mỗi lớp:", chain_best(groups, 3),
      "| chọn (A, C, B) trực tiếp:", chain_cost(groups, ["A", "C", "B"]))

top-2 mỗi lớp: 8 | top-3 mỗi lớp: 6 | chọn (A, C, B) trực tiếp: 6


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đếm tần suất, thử mọi `x`, ghép với `y` tốt nhất khác `x`

In [7]:
def min_ops_each_x(nums):
    n = len(nums)
    cE, cO = Counter(nums[0::2]), Counter(nums[1::2])
    best_odd = top2(cO)
    keep = 0
    for x, fx in list(cE.items()) + [(0, 0)]:             # x = 0 đại diện "gán số mới"
        for y, gy in best_odd:
            if y != x:                                    # y tốt nhất khác x
                keep = max(keep, fx + gy); break
    return n - keep

for nums_, exp in examples:
    print(min_ops_each_x(nums_), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )


**Lưu ý:** khi `x = 0` (số mới) và `y = 0` (cũng "số mới") ta vẫn coi được là khác nhau (hai số mới khác nhau), nhưng vì cả hai đều đóng góp 0, bỏ qua cặp này không mất gì. **Độ phức tạp:** `O(n)` đếm + `O(V)` duyệt.

### 7.2 Cách 3 — Chỉ top-2 mỗi nhóm (công thức)

In [8]:
def min_ops_top2(nums):
    n = len(nums)
    (a1, f1), (a2, f2) = top2(Counter(nums[0::2]))
    (b1, g1), (b2, g2) = top2(Counter(nums[1::2]))
    if a1 != b1:
        return n - f1 - g1
    return n - max(f1 + g2, f2 + g1)

for nums_, exp in examples:
    print(min_ops_top2(nums_), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )


**Chú ý trường hợp đệm:** nếu cả hai nhóm chỉ có giá trị đệm `0` (không thể, vì nhóm chẵn luôn có ≥ 1 phần tử). Nếu nhóm lẻ rỗng (`n = 1`): `b1 = 0 ≠ a1` ⇒ `1 − 1 − 0 = 0` ✓.

### 7.3 Cách 4 — Sắp danh sách tần suất giảm dần và thử 4 tổ hợp

Không dùng `most_common`: sắp `items()` theo tần suất giảm, lấy hai đầu, thử 4 cặp `(aᵢ, bⱼ)` với `i, j ∈ {1, 2}` và `aᵢ ≠ bⱼ`. Dễ tổng quát hóa thành "thử top-k mỗi bên".

In [9]:
def min_ops_sorted(nums):
    n = len(nums)
    def top(lst):
        items = sorted(Counter(lst).items(), key=lambda kv: -kv[1])[:2]
        return items + [(-1 - i, 0) for i in range(2 - len(items))]   # giá trị giả khác nhau, âm
    E, O = top(nums[0::2]), top(nums[1::2])
    keep = max(fe + fo for (ve, fe) in E for (vo, fo) in O if ve != vo)
    return n - keep

for nums_, exp in examples:
    print(min_ops_sorted(nums_), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )


**Độ phức tạp:** `O(n + V log V)`.

### 7.4 Cách 5 — Một lượt đếm bằng mảng, tìm top-2 tuyến tính

Giá trị `≤ 10⁵` ⇒ đếm bằng **mảng** (nhanh hơn `Counter`), rồi tìm top-2 bằng một vòng quét (như tìm số lớn nhất và nhì). Không sắp xếp.

In [10]:
def min_ops_array(nums):
    n = len(nums)
    M = max(nums) + 1
    cE = [0] * M; cO = [0] * M
    for i, v in enumerate(nums):
        if i & 1: cO[v] += 1
        else: cE[v] += 1
    def top2_arr(c):
        v1 = v2 = 0; f1 = f2 = 0                    # giá trị 0 không có trong nums
        for v in range(1, M):
            f = c[v]
            if f > f1:
                v2, f2 = v1, f1; v1, f1 = v, f
            elif f > f2:
                v2, f2 = v, f
        return v1, f1, v2, f2
    a1, f1, a2, f2 = top2_arr(cE)
    b1, g1, b2, g2 = top2_arr(cO)
    if a1 != b1:
        return n - f1 - g1
    return n - max(f1 + g2, f2 + g1)

for nums_, exp in examples:
    print(min_ops_array(nums_), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )


**Độ phức tạp:** `O(n + max(nums))`, bộ nhớ `O(max(nums))`. Khi nhóm lẻ rỗng, `top2_arr` trả `(0, 0, 0, 0)`; `a1 ≥ 1 ≠ 0 = b1` ⇒ đúng.

**Bẫy:** nếu **cả hai** nhóm đều có `v1 = 0` (không thể) thì công thức coi là "trùng". Và khi `a1 = b1 = 0`… không xảy ra vì nhóm chẵn không rỗng.

## 8. TRACE từng bước

### 8.1 Trace cách 3 trên hai ví dụ của đề

In [11]:
def trace_top2(nums):
    n = len(nums)
    cE, cO = Counter(nums[0::2]), Counter(nums[1::2])
    (a1, f1), (a2, f2) = top2(cE); (b1, g1), (b2, g2) = top2(cO)
    print(f"  nums={nums}")
    print(f"    chẵn: {dict(cE)} -> top2 = ({a1}:{f1}), ({a2}:{f2})")
    print(f"    lẻ  : {dict(cO)} -> top2 = ({b1}:{g1}), ({b2}:{g2})")
    if a1 != b1:
        keep = f1 + g1
        print(f"    a1={a1} ≠ b1={b1} -> giữ f1+g1 = {keep}")
    else:
        keep = max(f1 + g2, f2 + g1)
        print(f"    a1 = b1 = {a1} (xung đột) -> giữ max(f1+g2={f1 + g2}, f2+g1={f2 + g1}) = {keep}")
    print(f"    => thao tác = {n} - {keep} = {n - keep}")
    return n - keep

for nums_, _ in examples:
    trace_top2(nums_)

  nums=[3, 1, 3, 2, 4, 3]
    chẵn: {3: 2, 4: 1} -> top2 = (3:2), (4:1)
    lẻ  : {1: 1, 2: 1, 3: 1} -> top2 = (1:1), (2:1)
    a1=3 ≠ b1=1 -> giữ f1+g1 = 3
    => thao tác = 6 - 3 = 3
  nums=[1, 2, 2, 2, 2]
    chẵn: {1: 1, 2: 2} -> top2 = (2:2), (1:1)
    lẻ  : {2: 2} -> top2 = (2:2), (0:0)
    a1 = b1 = 2 (xung đột) -> giữ max(f1+g2=2, f2+g1=3) = 3
    => thao tác = 5 - 3 = 2


### 8.2 Trace các trường hợp đặc biệt (ví dụ tự dựng — đã đối chiếu vét cạn, đối chiếu lại đề gốc)

In [12]:
for nums_ in [[7], [2, 2], [1, 1, 1, 1, 1], [5, 9, 5, 9], [4, 4, 4, 3, 3, 3]]:
    r = trace_top2(nums_)
    assert r == min_ops_brute(nums_)

  nums=[7]
    chẵn: {7: 1} -> top2 = (7:1), (0:0)
    lẻ  : {} -> top2 = (0:0), (0:0)
    a1=7 ≠ b1=0 -> giữ f1+g1 = 1
    => thao tác = 1 - 1 = 0
  nums=[2, 2]
    chẵn: {2: 1} -> top2 = (2:1), (0:0)
    lẻ  : {2: 1} -> top2 = (2:1), (0:0)
    a1 = b1 = 2 (xung đột) -> giữ max(f1+g2=1, f2+g1=1) = 1
    => thao tác = 2 - 1 = 1
  nums=[1, 1, 1, 1, 1]
    chẵn: {1: 3} -> top2 = (1:3), (0:0)
    lẻ  : {1: 2} -> top2 = (1:2), (0:0)
    a1 = b1 = 1 (xung đột) -> giữ max(f1+g2=3, f2+g1=2) = 3
    => thao tác = 5 - 3 = 2
  nums=[5, 9, 5, 9]
    chẵn: {5: 2} -> top2 = (5:2), (0:0)
    lẻ  : {9: 2} -> top2 = (9:2), (0:0)
    a1=5 ≠ b1=9 -> giữ f1+g1 = 4
    => thao tác = 4 - 4 = 0
  nums=[4, 4, 4, 3, 3, 3]
    chẵn: {4: 2, 3: 1} -> top2 = (4:2), (3:1)
    lẻ  : {4: 1, 3: 2} -> top2 = (3:2), (4:1)
    a1=4 ≠ b1=3 -> giữ f1+g1 = 4
    => thao tác = 6 - 4 = 2


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def minimumOperations(self, nums: List[int]) -> int:
        n = len(nums)

        def best_two(values):
            top = Counter(values).most_common(2)
            while len(top) < 2:
                top.append((0, 0))              # 0 không phải số dương -> "giá trị mới"
            return top

        (a1, f1), (_, f2) = best_two(nums[0::2])
        (b1, g1), (_, g2) = best_two(nums[1::2])
        if a1 != b1:
            return n - f1 - g1
        return n - max(f1 + g2, f2 + g1)

for nums_, exp in examples:
    assert Solution().minimumOperations(nums_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Rút gọn bài toán.** Mảng xen kẽ ⇔ ∃ `x ≠ y` (số dương): `nums[i] = x` với `i` chẵn, `= y` với `i` lẻ (khi `n = 1` chỉ cần `x`). Đổi `nums[i]` tốn 1 thao tác nếu nó khác giá trị đích, 0 nếu bằng. ⇒ thao tác `= n − cntE[x] − cntO[y]`.
2. **Giá trị mới.** `cntE[x] = 0` với `x` không có trong nhóm chẵn; ta mô hình bằng giá trị `0` với tần suất 0. Luôn chọn được số dương mới khác `y` thật.
3. **Bổ đề top-2** (mục 6) cho `max_{x≠y} (cntE[x] + cntO[y])` từ 4 số `f₁, f₂, g₁, g₂`.
4. **Hòa tần suất.** Nếu nhóm chẵn có hai giá trị cùng `f₁` thì `most_common` chọn một trong hai làm `a₁`; nếu `a₁ = b₁` thì `f₂ = f₁` và công thức cho `f₁ + g₁` — vẫn đúng (chọn giá trị còn lại). Mục 12 kiểm nhiều trường hợp hòa.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `V` = số giá trị khác nhau, `A = max(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp `(x, y)` | `O(V² n)` | `O(V)` |
| 2. Mọi `x` + `y` tốt nhất | `O(n + V)` | `O(V)` |
| 3. Top-2 công thức | `O(n + V)` (`most_common(2)` dùng heap) | `O(V)` |
| 4. Sắp tần suất | `O(n + V log V)` | `O(V)` |
| 5. Mảng đếm | `O(n + A)` | `O(A)` |

In [14]:
rnd = random.Random(2170)
big = [rnd.randint(1, 10**5) for _ in range(10**5)]
big2 = [rnd.randint(1, 3) for _ in range(10**5)]
for name, fn in [("each_x", min_ops_each_x), ("top2", min_ops_top2), ("sorted", min_ops_sorted),
                 ("array", min_ops_array), ("Solution", Solution().minimumOperations)]:
    t0 = time.perf_counter(); r1 = fn(big); r2 = fn(big2); dt = time.perf_counter() - t0
    print(f"{name:<9}: {r1}, {r2}  ({dt * 1000:6.1f} ms cho 2 mảng 1e5)")

each_x   : 99988, 66448  (  21.2 ms cho 2 mảng 1e5)
top2     : 99988, 66448  (  12.0 ms cho 2 mảng 1e5)
sorted   : 99988, 66448  (  20.3 ms cho 2 mảng 1e5)


array    : 99988, 66448  (  20.0 ms cho 2 mảng 1e5)
Solution : 99988, 66448  (  11.2 ms cho 2 mảng 1e5)


## 12. Bộ test

In [15]:
ALL = {
    "each_x": min_ops_each_x,
    "top2": min_ops_top2,
    "sorted": min_ops_sorted,
    "array": min_ops_array,
    "Solution": lambda a: Solution().minimumOperations(a),
}

def is_alternating(a):
    return all(a[i - 2] == a[i] for i in range(2, len(a))) and all(a[i - 1] != a[i] for i in range(1, len(a)))

def check(nums, exp=None, brute=True):
    ref = min_ops_brute(nums) if brute else min_ops_top2(nums)
    for name, fn in ALL.items():
        got = fn(list(nums))
        assert got == ref, (name, nums, got, ref)
    if exp is not None:
        assert ref == exp, (nums, ref, exp)
    return ref

for nums_, exp in examples:
    check(nums_, exp)
print("Ví dụ đề: OK")

check([7], 0)                      # n = 1
check([2, 2], 1)                   # hai phần tử bằng nhau
check([1, 2], 0)                   # đã xen kẽ
check([1, 1, 1, 1, 1], 2)          # toàn bằng: đổi các vị trí lẻ
check([5, 9, 5, 9], 0)
check([4, 4, 4, 3, 3, 3], 2)       # chẵn [4,4,3], lẻ [4,3,3] -> x=4, y=3
check([1, 2, 1, 2, 2, 1, 2, 1], 4) # hai nhóm hòa tần suất
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(3000):
    n = rnd.randint(1, 9)
    check([rnd.randint(1, 3) for _ in range(n)])
print("3000 mảng nhỏ (giá trị 1..3, nhiều hòa/xung đột) so với vét cạn: OK")

for _ in range(20):
    n = rnd.randint(1, 10**5)
    check([rnd.randint(1, rnd.choice([2, 5, 10**5])) for _ in range(n)], brute=False)
print("20 mảng lớn (đối chiếu chéo): OK")

Ví dụ đề: OK
Edge case: OK


3000 mảng nhỏ (giá trị 1..3, nhiều hòa/xung đột) so với vét cạn: OK


20 mảng lớn (đối chiếu chéo): OK


In [16]:
# Validator xây dựng: dựng mảng xen kẽ cụ thể đạt đúng số thao tác
def construct(nums):
    n = len(nums)
    cE, cO = Counter(nums[0::2]), Counter(nums[1::2])
    fresh1, fresh2 = max(nums) + 1, max(nums) + 2
    best = None
    for x in list(cE) + [fresh1]:
        for y in list(cO) + [fresh2]:
            if x != y:
                keep = cE.get(x, 0) + cO.get(y, 0)
                if best is None or keep > best[0]:
                    best = (keep, x, y)
    _, x, y = best
    return [x if i % 2 == 0 else y for i in range(n)]

rnd = random.Random(8)
for _ in range(500):
    a = [rnd.randint(1, 4) for _ in range(rnd.randint(1, 12))]
    b = construct(a)
    assert is_alternating(b) and sum(u != v for u, v in zip(a, b)) == Solution().minimumOperations(a)
print("Validator: luôn dựng được mảng xen kẽ đạt đúng số thao tác của Solution")

Validator: luôn dựng được mảng xen kẽ đạt đúng số thao tác của Solution


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lấy giá trị phổ biến nhất mỗi nhóm, quên trường hợp trùng nhau

In [17]:
def bug_ignore_conflict(nums):
    n = len(nums)
    (a1, f1), = Counter(nums[0::2]).most_common(1)
    g1 = Counter(nums[1::2]).most_common(1)[0][1] if n > 1 else 0
    return n - f1 - g1

print("[1,2,2,2,2] -> sai:", bug_ignore_conflict([1, 2, 2, 2, 2]), "| đúng:", Solution().minimumOperations([1, 2, 2, 2, 2]))

[1,2,2,2,2] -> sai: 1 | đúng: 2


**Vì sao sai:** cả hai nhóm chọn `2` ⇒ mảng `[2,2,2,2,2]` không xen kẽ. **Sửa:** khi `a₁ = b₁`, so `f₁ + g₂` với `f₂ + g₁`.

### Lỗi 2 — Khi trùng, chỉ đổi nhóm **lẻ** sang giá trị nhì (không thử đổi nhóm chẵn)

In [18]:
def bug_only_change_odd(nums):
    n = len(nums)
    (a1, f1), (a2, f2) = top2(Counter(nums[0::2]))
    (b1, g1), (b2, g2) = top2(Counter(nums[1::2]))
    if a1 != b1:
        return n - f1 - g1
    return n - (f1 + g2)                     # SAI: thiếu f2 + g1

nums_ = [1, 2, 2, 2, 2]
print("sai :", bug_only_change_odd(nums_), "| đúng:", Solution().minimumOperations(nums_))

sai : 3 | đúng: 2


**Vì sao sai:** ở ví dụ 2, nên giữ nhóm lẻ (`2` hai lần) và đổi nhóm chẵn sang `1`. **Sửa:** `max(f₁ + g₂, f₂ + g₁)`.

### Lỗi 3 — Không đệm top-2 ⇒ thiếu phương án "gán số mới"

In [19]:
def bug_no_padding(nums):
    n = len(nums)
    tE = Counter(nums[0::2]).most_common(2); tO = Counter(nums[1::2]).most_common(2)
    keep = 0
    for ve, fe in tE:
        for vo, fo in tO:
            if ve != vo:
                keep = max(keep, fe + fo)    # SAI: không có lựa chọn "nhóm kia gán số mới"
    return n - keep

nums_ = [1, 1, 1, 1, 1]
print("sai :", bug_no_padding(nums_), "| đúng:", Solution().minimumOperations(nums_))

sai : 5 | đúng: 2


**Vì sao sai:** mỗi nhóm chỉ có giá trị `1` ⇒ không có cặp nào khác nhau ⇒ `keep = 0`, trả `5`. Thực ra giữ nhóm chẵn (3 phần tử) và gán nhóm lẻ số mới: `2`. **Sửa:** đệm `(0, 0)`.

### Lỗi 4 — Bỏ qua điều kiện `x ≠ y` hoàn toàn (đếm "giá trị phổ biến nhất toàn mảng")

In [20]:
def bug_global_mode(nums):
    return len(nums) - Counter(nums).most_common(1)[0][1]

nums_ = [3, 1, 3, 2, 4, 3]
print("sai :", bug_global_mode(nums_), "| đúng:", Solution().minimumOperations(nums_))
nums_ = [1, 2, 1, 2]
print("sai :", bug_global_mode(nums_), "| đúng:", Solution().minimumOperations(nums_))

sai : 3 | đúng: 3
sai : 2 | đúng: 0


**Vì sao sai:** nhầm "xen kẽ" với "mọi phần tử bằng nhau" — đó là trường hợp **bị cấm**.

### Lỗi 5 — Gõ nhầm công thức khi xung đột: `max(f₁ + f₂, g₁ + g₂)`

In [21]:
def bug_formula_typo(nums):
    n = len(nums)
    (a1, f1), (a2, f2) = top2(Counter(nums[0::2]))
    (b1, g1), (b2, g2) = top2(Counter(nums[1::2]))
    if a1 != b1:
        return n - f1 - g1
    return n - max(f1 + f2, g1 + g2)          # SAI: cộng hai giá trị của CÙNG một nhóm

nums_ = [2, 2, 3, 2]
print("sai :", bug_formula_typo(nums_), "| đúng:", Solution().minimumOperations(nums_))

sai : 2 | đúng: 1


**Vì sao sai:** `f₁ + f₂` nghĩa là "giữ hai giá trị khác nhau trong **cùng** nhóm chẵn" — vô nghĩa, vì cả nhóm phải bằng một giá trị. Phải ghép **chéo**: giá trị của nhóm chẵn với giá trị của nhóm lẻ. Ví dụ này được tìm bằng cách quét ngẫu nhiên so với vét cạn — công thức sai vẫn qua cả hai ví dụ của đề! **Sửa:** `max(f₁ + g₂, f₂ + g₁)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Ràng buộc kiểu "các vị trí cùng lớp phải bằng nhau" ⇒ tách lớp (chẵn/lẻ, `i mod k`), đếm tần suất từng lớp.
- "Hai lớp kề nhau phải khác" ⇒ chỉ cần **top-2** mỗi lớp.
- Tối thiểu thao tác = `n − tối đa phần tử giữ lại`.
- Đệm top-2 bằng giá trị không thể xuất hiện, tần suất 0.
- Test: `n = 1`, `n = 2` hai số bằng nhau, toàn mảng một giá trị, hai lớp cùng mode, hòa tần suất.

## 15. Biến thể của bài

1. **Chu kỳ `k`** (`nums[i] == nums[i − k]`, và các lớp **kề nhau** khác nhau): DP trên `k` lớp với top-2 mỗi lớp — bài tập 5 (bản vòng `i mod k`).
2. **Chỉ yêu cầu `nums[i−2] == nums[i]`** (không cần khác nhau): mỗi lớp lấy mode độc lập.
3. **LeetCode 1888 — Minimum Number of Flips to Make the Binary String Alternating.**
4. **LeetCode 2033 — Minimum Operations to Make a Uni-Value Grid.**
5. **Chi phí đổi khác nhau theo vị trí** (trọng số): thay tần suất bằng tổng trọng số.

In [22]:
def min_ops_weighted(nums, w):
    """Biến thể 5: đổi nums[i] tốn w[i]; tối thiểu tổng chi phí. Thay 'tần suất' bằng 'tổng trọng số giữ lại'."""
    from collections import defaultdict
    sE, sO = defaultdict(int), defaultdict(int)
    for i, (v, c) in enumerate(zip(nums, w)):
        (sO if i & 1 else sE)[v] += c
    def t2(d):
        top = sorted(d.items(), key=lambda kv: -kv[1])[:2]
        return top + [(0, 0)] * (2 - len(top))
    (a1, f1), (a2, f2) = t2(sE); (b1, g1), (b2, g2) = t2(sO)
    keep = f1 + g1 if a1 != b1 else max(f1 + g2, f2 + g1)
    return sum(w) - keep

def weighted_brute(nums, w):
    m = max(nums); cand = set(nums) | {m + 1, m + 2}; best = None
    for x in cand:
        for y in cand:
            if x != y or len(nums) == 1:
                c = sum(wi for i, (v, wi) in enumerate(zip(nums, w)) if v != (x if i % 2 == 0 else y))
                best = c if best is None else min(best, c)
    return best

rnd_v = random.Random(15)
for _ in range(1000):
    n = rnd_v.randint(1, 8)
    a = [rnd_v.randint(1, 3) for _ in range(n)]; w = [rnd_v.randint(1, 9) for _ in range(n)]
    assert min_ops_weighted(a, w) == weighted_brute(a, w)
print("Biến thể có trọng số: khớp vét cạn; ví dụ", min_ops_weighted([1, 2, 2, 2, 2], [5, 1, 1, 1, 1]))

Biến thể có trọng số: khớp vét cạn; ví dụ 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp | thử `(x, y)` | `O(V² n)` | `O(V)` | kiểm thử |
| 2. Mọi `x` | `y` tốt nhất ≠ `x` | `O(n + V)` | `O(V)` | trực giác dễ |
| 3. Top-2 | công thức 2 trường hợp | `O(n + V)` | `O(V)` | **nộp bài** |
| 4. Sắp tần suất | 4 tổ hợp | `O(n + V log V)` | `O(V)` | tổng quát top-k |
| 5. Mảng đếm | top-2 tuyến tính | `O(n + A)` | `O(A)` | ngôn ngữ khác / nhanh |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minimumOperations([2, 2, 2, 2])`?

**Bài 2 (tính tay):** `nums = [1, 3, 1, 3, 3, 1, 3, 1]`. Tính `cntE`, `cntO`, top-2 và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `make_alternating(nums)` trả về **một** mảng xen kẽ đạt số thao tác tối thiểu (validator ở mục 12).

**Bài 4 (tự viết lại):** Viết lại cách 3 **không dùng `Counter`** (dùng `dict` và vòng lặp tìm top-2).

**Bài 5 (mini challenge):** Biến thể chu kỳ `k` dạng vòng: các lớp `0..k−1` (`i mod k`), mọi phần tử cùng lớp phải bằng nhau, lớp `j` và `j + 1` (và lớp `k−1` với lớp `0`… chỉ xét các lớp kề nhau **tuyến tính** `j`, `j+1`) phải khác nhau. Tối thiểu số thao tác. Gợi ý: DP trên lớp, trạng thái = giá trị chọn cho lớp trước, chỉ giữ một số ít ứng viên mỗi lớp (bao nhiêu là đủ? xem gợi ý mục 18).

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def make_alternating(nums):
    # TODO: chọn (x, y) theo top-2; giá trị mới = max(nums) + 1 (hoặc + 2)
    pass

# Bài 4 — TODO
def min_ops_no_counter(nums):
    # TODO: dict đếm; quét tìm (v1, f1), (v2, f2)
    pass

# Bài 5 — TODO
def min_ops_period_k(nums, k):
    # TODO: DP trên lớp j = 0..k-1, ứng viên mỗi lớp: vài giá trị phổ biến nhất + "mới"
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Hai nhóm đều toàn `2`.
- Mức 2: Xung đột; `f₂ = g₂ = 0`.
- Mức 3: `4 − max(2 + 0, 0 + 2) = 2`.

**Bài 2**
- Mức 1: Chẵn `[1, 1, 3, 3]`, lẻ `[3, 3, 1, 1]`.
- Mức 2: Mỗi nhóm hòa `{1: 2, 3: 2}`.
- Mức 3: Chọn `x = 1, y = 3` (hoặc ngược lại) giữ 4 ⇒ `4`.

**Bài 3**
- Mức 1: Tính lại `(x, y)` tốt nhất, thay giá trị đệm `0` bằng số mới.
- Mức 2: Nếu cả `x` và `y` đều là "mới", dùng hai số mới khác nhau.
- Mức 3: Kiểm bằng `is_alternating` + đếm số vị trí đổi.

**Bài 4**
- Mức 1: `d[v] = d.get(v, 0) + 1`.
- Mức 2: Quét `d.items()` giữ hai cặp tốt nhất.
- Mức 3: Đối chiếu `Solution`.

**Bài 5**
- Mức 1: Chi phí lớp `j` nếu chọn giá trị `v` = `size_j − cnt_j[v]`.
- Mức 2: Lớp ở giữa có **hai** lớp kề, có thể "chặn" hai giá trị tốt nhất của nó ⇒ cần ứng viên **top-3** + "mới" (top-2 chỉ đủ khi mỗi lớp có một hàng xóm, như bài gốc `k = 2`). Đây là cái bẫy: bản top-2 sai — đã kiểm bằng vét cạn.
- Mức 3: `dp[v]` = chi phí nhỏ nhất tới lớp `j` khi lớp `j` chọn `v`; chuyển từ lớp trước với ràng buộc khác giá trị. Đối chiếu vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1
r1 = Solution().minimumOperations([2, 2, 2, 2])
print("Bài 1:", r1); assert r1 == 2 == min_ops_brute([2, 2, 2, 2])

# Bài 2
b2 = [1, 3, 1, 3, 3, 1, 3, 1]
print("Bài 2:"); r2 = trace_top2(b2); assert r2 == 4 == min_ops_brute(b2)

# Bài 3
def make_alternating_sol(nums):
    n = len(nums); fresh = [max(nums) + 1, max(nums) + 2]
    (a1, f1), (a2, f2) = top2(Counter(nums[0::2])); (b1, g1), (b2, g2) = top2(Counter(nums[1::2]))
    if a1 != b1: x, y = a1, b1
    elif f1 + g2 >= f2 + g1: x, y = a1, b2
    else: x, y = a2, b1
    if x == 0: x = fresh[0]
    if y == 0: y = fresh[1] if x == fresh[0] else fresh[0]
    if x == y: y = fresh[1]
    return [x if i % 2 == 0 else y for i in range(n)]

rnd = random.Random(3)
for _ in range(2000):
    a = [rnd.randint(1, 3) for _ in range(rnd.randint(1, 10))]
    b = make_alternating_sol(a)
    assert is_alternating(b) and min(b) >= 1 and sum(u != v for u, v in zip(a, b)) == Solution().minimumOperations(a)
print("Bài 3 OK, ví dụ:", make_alternating_sol([1, 2, 2, 2, 2]), make_alternating_sol([3, 1, 3, 2, 4, 3]))

Bài 1: 2
Bài 2:
  nums=[1, 3, 1, 3, 3, 1, 3, 1]
    chẵn: {1: 2, 3: 2} -> top2 = (1:2), (3:2)
    lẻ  : {3: 2, 1: 2} -> top2 = (3:2), (1:2)
    a1=1 ≠ b1=3 -> giữ f1+g1 = 4
    => thao tác = 8 - 4 = 4
Bài 3 OK, ví dụ: [1, 2, 1, 2, 1] [3, 1, 3, 1, 3, 1]


In [25]:
# Bài 4
def min_ops_no_counter_sol(nums):
    def t2(vals):
        d = {}
        for v in vals:
            d[v] = d.get(v, 0) + 1
        v1 = v2 = 0; f1 = f2 = 0
        for v, f in d.items():
            if f > f1: v2, f2, v1, f1 = v1, f1, v, f
            elif f > f2: v2, f2 = v, f
        return v1, f1, v2, f2
    a1, f1, _, f2 = t2(nums[0::2]); b1, g1, _, g2 = t2(nums[1::2])
    n = len(nums)
    return n - (f1 + g1 if a1 != b1 else max(f1 + g2, f2 + g1))

# Bài 5
from itertools import product as iproduct

def min_ops_period_k_sol(nums, k):
    groups = [nums[j::k] for j in range(k) if nums[j::k]]
    NEW = [-1, -2]                                  # hai "giá trị mới" khác nhau, khác mọi số dương
    prev = None                                     # dict giá trị -> chi phí nhỏ nhất
    for g in groups:
        c = Counter(g)
        cand = [v for v, _ in c.most_common(3)] + NEW   # lớp giữa có HAI hàng xóm -> cần top-3
        cur = {}
        for v in cand:
            cost = len(g) - c.get(v, 0)
            if prev is None:
                cur[v] = cost
            else:
                cur[v] = cost + min(pc for pv, pc in prev.items() if pv != v)
        prev = cur
    return min(prev.values())

def period_k_brute(nums, k):
    groups = [nums[j::k] for j in range(k) if nums[j::k]]
    vals = sorted(set(nums)) + [max(nums) + 1, max(nums) + 2]
    best = None
    for choice in iproduct(vals, repeat=len(groups)):
        if any(choice[j] == choice[j + 1] for j in range(len(groups) - 1)):
            continue
        c = sum(sum(1 for v in g if v != ch) for g, ch in zip(groups, choice))
        best = c if best is None else min(best, c)
    return best

rnd = random.Random(5)
for _ in range(600):
    a = [rnd.randint(1, 3) for _ in range(rnd.randint(1, 10))]
    assert min_ops_no_counter_sol(a) == Solution().minimumOperations(a)
    k = rnd.randint(1, 4)
    assert min_ops_period_k_sol(a, k) == period_k_brute(a, k)
    if k == 2:
        assert min_ops_period_k_sol(a, 2) == Solution().minimumOperations(a)
print("Bài 4, Bài 5 OK; ví dụ chu kỳ 3:", min_ops_period_k_sol([1, 2, 3, 1, 2, 2, 1, 1, 3], 3))

Bài 4, Bài 5 OK; ví dụ chu kỳ 3: 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
def best_two(vals):
    top = Counter(vals).most_common(2)
    return top + [(0, 0)] * (2 - len(top))
(a1, f1), (_, f2) = best_two(nums[0::2])
(b1, g1), (_, g2) = best_two(nums[1::2])
keep = f1 + g1 if a1 != b1 else max(f1 + g2, f2 + g1)
return len(nums) - keep
```

**Checklist:**
- [ ] Xử lý `a₁ == b₁` bằng `max(f₁ + g₂, f₂ + g₁)` (thử đổi **cả hai** phía)?
- [ ] Đệm top-2 bằng giá trị **không xuất hiện** tần suất 0?
- [ ] `n = 1` trả `0`?
- [ ] Không nhầm "xen kẽ" với "tất cả bằng nhau"?
- [ ] Đã đối chiếu vét cạn trên mảng giá trị nhỏ (nhiều xung đột)?
- [ ] Chữ ký đúng: `def minimumOperations(self, nums: List[int]) -> int`?